<a href="https://colab.research.google.com/github/krishnasivaprasadm-jpg/cardiovascular_CA-SAE-AFB-a/blob/main/CA_SAE_AFB_Baselines_NestedCV(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Baseline Models — Nested CV

Google Colab-ready implementation.


In [1]:

# Baseline Models — Nested 5x3 CV
# Proposed CA-SAE-AFB model is intentionally NOT included.
# Upload heart.csv before running.

!pip -q install xgboost openpyxl

import os, random
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, precision_score,
                             recall_score, f1_score, matthews_corrcoef, roc_auc_score,
                             average_precision_score)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (RandomForestClassifier, ExtraTreesClassifier,
                              GradientBoostingClassifier, AdaBoostClassifier)
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier
import xgboost as xgb

SEED=2026
OUTER_FOLDS=5
INNER_FOLDS=3

df=pd.read_csv("heart.csv")
TARGET="HeartDisease"
X=df.drop(columns=[TARGET]).copy()
y=df[TARGET].astype(int).to_numpy()
X=pd.get_dummies(X,drop_first=True).replace([np.inf,-np.inf],np.nan)

models={
"Logistic Regression": LogisticRegression(max_iter=2000,class_weight="balanced",random_state=SEED),
"Random Forest": RandomForestClassifier(n_estimators=300,max_depth=None,class_weight="balanced",
                                         random_state=SEED,n_jobs=-1),
"Extra Trees": ExtraTreesClassifier(n_estimators=300,class_weight="balanced",random_state=SEED,n_jobs=-1),
"Gradient Boosting": GradientBoostingClassifier(n_estimators=200,learning_rate=.05,max_depth=2,random_state=SEED),
"AdaBoost": AdaBoostClassifier(n_estimators=200,learning_rate=.05,random_state=SEED),
"XGBoost": xgb.XGBClassifier(n_estimators=250,max_depth=3,learning_rate=.04,
                             subsample=.85,colsample_bytree=.85,eval_metric="logloss",
                             random_state=SEED,n_jobs=-1),
"SVM-RBF": SVC(C=1.0,kernel="rbf",probability=True,class_weight="balanced",random_state=SEED),
"KNN": KNeighborsClassifier(n_neighbors=7),
"Decision Tree": DecisionTreeClassifier(max_depth=5,class_weight="balanced",random_state=SEED),
"MLP": MLPClassifier(hidden_layer_sizes=(64,32),max_iter=1000,early_stopping=True,
                     random_state=SEED)
}

def preprocess_fit(A,B):
    imp=SimpleImputer(strategy="median")
    sc=StandardScaler()
    A=imp.fit_transform(A); B=imp.transform(B)
    A=sc.fit_transform(A); B=sc.transform(B)
    return A,B

def best_threshold(y,p):
    grid=np.arange(.20,.81,.01)
    return float(grid[np.argmax([f1_score(y,(p>=t).astype(int)) for t in grid])])

def evaluate(y,p,t):
    z=(p>=t).astype(int)
    return dict(Accuracy=accuracy_score(y,z),
                Balanced_Accuracy=balanced_accuracy_score(y,z),
                Precision=precision_score(y,z,zero_division=0),
                Recall=recall_score(y,z,zero_division=0),
                F1=f1_score(y,z,zero_division=0),
                MCC=matthews_corrcoef(y,z),
                ROC_AUC=roc_auc_score(y,p),
                PR_AUC=average_precision_score(y,p))

outer=StratifiedKFold(OUTER_FOLDS,shuffle=True,random_state=SEED)
all_results=[]

for name, model in models.items():
    for fold,(tr,te) in enumerate(outer.split(X,y),1):
        Xtr0,Xte0=X.iloc[tr],X.iloc[te]
        ytr,yte=y[tr],y[te]

        inner=StratifiedKFold(INNER_FOLDS,shuffle=True,random_state=SEED+fold)
        oof=np.zeros(len(tr))

        for itr,iva in inner.split(Xtr0,ytr):
            A,B=preprocess_fit(Xtr0.iloc[itr],Xtr0.iloc[iva])
            m=model.__class__(**model.get_params())
            m.fit(A,ytr[itr])
            oof[iva]=m.predict_proba(B)[:,1]

        threshold=best_threshold(ytr,oof)

        A,B=preprocess_fit(Xtr0,Xte0)
        m=model.__class__(**model.get_params())
        m.fit(A,ytr)
        p=m.predict_proba(B)[:,1]

        row=evaluate(yte,p,threshold)
        row.update(Model=name,Fold=fold,Threshold=threshold)
        all_results.append(row)

results=pd.DataFrame(all_results)
print(results)
summary=results.groupby("Model").agg(["mean","std"])
print("\nSummary:")
print(summary)

results.to_csv("baseline_nestedCV_results.csv",index=False)
summary.to_csv("baseline_nestedCV_summary.csv")
print("\nSaved baseline_nestedCV_results.csv and baseline_nestedCV_summary.csv")


    Accuracy  Balanced_Accuracy  Precision    Recall        F1       MCC  \
0   0.826087           0.815638   0.801724  0.911765  0.853211  0.650045   
1   0.880435           0.875418   0.870370  0.921569  0.895238  0.757938   
2   0.853261           0.850909   0.864078  0.872549  0.868293  0.702700   
3   0.857923           0.848346   0.826087  0.940594  0.879630  0.716976   
4   0.874317           0.872374   0.882353  0.891089  0.886700  0.745649   
5   0.831522           0.820540   0.803419  0.921569  0.858447  0.662200   
6   0.875000           0.871712   0.876190  0.901961  0.888889  0.746510   
7   0.847826           0.841224   0.836364  0.901961  0.867925  0.691775   
8   0.885246           0.876539   0.850877  0.960396  0.902326  0.772736   
9   0.890710           0.890667   0.909091  0.891089  0.900000  0.779734   
10  0.847826           0.837637   0.818966  0.931373  0.871560  0.695351   
11  0.869565           0.864419   0.861111  0.911765  0.885714  0.735731   
12  0.847826